# Notebook 14 — MJO Supervised 2D Encoder
**Project:** ENSO-BSISO SSL — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Supervised contrastive 2D encoder for MJO. Pairs defined by RMM phase + ENSO category — exact analog of `nb07c` (BSISO supervised 2D), now on global equatorial inputs.

## Setup (locked decisions)

| Knob | Value | Source |
|---|---|---|
| Input | `X_MJO.npy`, shape `(N, 3, 1, 180)` | nb13, Q1=A meridional avg |
| Channel order | `[u850, OLR, u200]` | nb13 |
| Embedding dim | 2 (no L2 norm) | Phase 1 lesson from BSISO |
| Loss | raw dot product InfoNCE | nb07c |
| Temperature τ | 0.5 | nb07c |
| Weight decay | 1e-4 | nb07c |
| Optimizer | Adam, lr=1e-3, cosine→1e-5 | nb07c |
| Epochs | 50 | nb07c |
| Architecture | Conv2d kernel=(1,3), pool=(1,2) | Q5 recommendation |
| Active MJO filter | amplitude ≥ 1.0 AND phase ∈ [1,8] | RMM convention |
| Year split | every 5th year held out | nb04/nb07c |

## Outputs

- `MJO/checkpoints/encoder_mjo_sup_final.pth`
- `MJO/checkpoints/training_history_mjo_sup.json`
- `MJO/results/sup/embeddings.npy`, `training_curves.png`, `embedding_2d_overview.png`, `radius_diagnostics.png`, `linear_probe_results.json`, `enso_displacement.png`, `mjo_sup_summary.md`

## Runtime
~30–45 min on T4 (larger N than BSISO: ~16,000 vs ~6,500).

---

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt

EMBEDDING_DIM = 2
RUN_TAG       = 'mjo_sup_fixed'
TEMPERATURE   = 0.07
EPOCHS        = 50
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
USE_VICREG    = True
VICREG_VAR    = 25.0
VICREG_COV    = 1.0

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR        = f'{PROJECT_DIR}/MJO'
PROCESSED_DIR  = f'{MJO_DIR}/data/processed'
CHECKPOINT_DIR = f'{MJO_DIR}/checkpoints'
RESULTS_DIR    = f'{MJO_DIR}/results/sup'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

X_FILE      = 'X_MJO.npy'
LABELS_FILE = 'labels_aligned_mjo.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:        {device}')
print(f'Embedding dim: {EMBEDDING_DIM} (no L2 normalization)')
print(f'Loss:          raw dot product InfoNCE (τ={TEMPERATURE})')
print(f'Run tag:       {RUN_TAG}')
print(f'Results dir:   MJO/results/sup/')

## Cell 2 — Load Data + Year-Based Split + Active MJO Filter

Build `phase_enso_index` only over **train days that are active MJO** (amplitude ≥ 1.0 AND phase ∈ {1,...,8}). Weak/inactive MJO days are kept in X but never sampled into positive/hard-negative pairs.

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])

print(f'X shape:  {X.shape}  (N, channels, lat, lon)')
print(f'Labels:   {len(labels)} rows')
assert X.shape[0] == len(labels), 'X / labels length mismatch!'
assert X.shape[1:] == (3, 1, 180), f'Unexpected X shape: {X.shape[1:]}; expected (3, 1, 180).'

# Year-based split: every 5th year held out
all_years = sorted(labels['date'].dt.year.unique())
val_years = all_years[::5]
train_years = [y for y in all_years if y not in val_years]
year_col  = labels['date'].dt.year
train_idx = labels.index[year_col.isin(train_years)].values
val_idx   = labels.index[year_col.isin(val_years)].values

print(f'\nVal years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)} samples ({100*len(train_idx)/len(labels):.1f}%)')
print(f'Val:   {len(val_idx)} samples ({100*len(val_idx)/len(labels):.1f}%)')

# Build (phase, ENSO) → indices ONLY from active MJO days in the train set
phase_enso_index = defaultdict(list)
for idx in train_idx:
    row = labels.loc[idx]
    if (not row['weak_mjo']) and (1 <= row['phase'] <= 8):
        key = (int(row['phase']), row['enso_category'])
        phase_enso_index[key].append(idx)

active_total = sum(len(v) for v in phase_enso_index.values())
print(f'\nActive MJO train days (ampl ≥ 1.0, phase 1-8): {active_total} / {len(train_idx)}')
print(f'\n(phase × ENSO) bin sizes:')
for ph in range(1, 9):
    counts = [len(phase_enso_index[(ph, c)]) for c in ['El Nino', 'Neutral', 'La Nina']]
    print(f'  P{ph}:  EN={counts[0]:4d}  Neu={counts[1]:4d}  LN={counts[2]:4d}')

## Cell 3 — PairSampler + Dataset + DataLoaders

Same logic as nb07c but column names changed to `phase` (RMM) instead of `bsiso_phase`.

In [ ]:
class PairSampler:
    def __init__(self, labels_df, phase_enso_index):
        self.labels = labels_df
        self.index  = phase_enso_index
        self.enso_categories = labels_df['enso_category'].unique().tolist()

    def sample_positive_pair(self):
        key = self._random_category()
        indices = self.index[key]
        if len(indices) < 2:
            return self.sample_easy_negative_pair()
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)
        # Prefer cross-year pairs if available
        year_A = self.labels.loc[idx_A, 'date'].year
        other = [i for i in indices if self.labels.loc[i, 'date'].year != year_A]
        if other:
            idx_B = np.random.choice(other)
        return idx_A, idx_B, 'positive'

    def sample_hard_negative_pair(self):
        if len(self.enso_categories) < 2:
            return self.sample_easy_negative_pair()
        phase = np.random.choice(range(1, 9))
        enso_A, enso_B = np.random.choice(self.enso_categories, size=2, replace=False)
        key_A, key_B = (phase, enso_A), (phase, enso_B)
        if not self.index[key_A] or not self.index[key_B]:
            return self.sample_positive_pair()
        idx_A = np.random.choice(self.index[key_A])
        idx_B = np.random.choice(self.index[key_B])
        return idx_A, idx_B, 'hard_negative'

    def sample_easy_negative_pair(self):
        phase_A, phase_B = np.random.choice(range(1, 9), size=2, replace=False)
        enso_A = np.random.choice(self.enso_categories)
        enso_B = np.random.choice(self.enso_categories)
        key_A, key_B = (phase_A, enso_A), (phase_B, enso_B)
        idx_A = (np.random.choice(self.index[key_A]) if self.index[key_A]
                 else self.labels[self.labels['phase'] == phase_A].sample(1).index[0])
        idx_B = (np.random.choice(self.index[key_B]) if self.index[key_B]
                 else self.labels[self.labels['phase'] == phase_B].sample(1).index[0])
        return idx_A, idx_B, 'easy_negative'

    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]


class MJOPairDataset(Dataset):
    def __init__(self, X_data, labels_df, phase_enso_index,
                 mode='train', train_indices=None):
        self.X       = X_data
        self.labels  = labels_df
        self.sampler = PairSampler(labels_df, phase_enso_index)
        self.mode    = mode
        self.train_indices = (train_indices if train_indices is not None
                              else np.arange(len(X_data)))
        if mode == 'val':
            self.val_pairs = self._create_val_pairs()

    def _create_val_pairs(self):
        pairs = []
        val_labels = self.labels.loc[self.train_indices]
        for phase in range(1, 9):
            for enso in self.sampler.enso_categories:
                group = val_labels[
                    (val_labels['phase'] == phase) &
                    (val_labels['enso_category'] == enso) &
                    (~val_labels['weak_mjo'])
                ].index.tolist()
                for i in range(len(group)):
                    for j in range(i + 1, len(group)):
                        pairs.append((group[i], group[j], 'positive'))
        return pairs[:1000]

    def __len__(self):
        return (len(self.train_indices) if self.mode == 'train'
                else len(self.val_pairs))

    def __getitem__(self, idx):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:
                idx_A, idx_B, _ = self.sampler.sample_positive_pair()
            elif r < 0.50:
                idx_A, idx_B, _ = self.sampler.sample_hard_negative_pair()
            else:
                idx_A, idx_B, _ = self.sampler.sample_easy_negative_pair()
        else:
            idx_A, idx_B, _ = self.val_pairs[idx]
        field_A = torch.from_numpy(self.X[idx_A]).float()
        field_B = torch.from_numpy(self.X[idx_B]).float()
        return field_A, field_B


train_dataset = MJOPairDataset(X, labels, phase_enso_index, mode='train', train_indices=train_idx)
val_dataset   = MJOPairDataset(X, labels, phase_enso_index, mode='val',   train_indices=val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_dataset)} items  →  {len(train_loader)} batches/epoch')
print(f'Val dataset:   {len(val_dataset)} pairs   →  {len(val_loader)} batches')

## Cell 4 — CNN Encoder for MJO Input `(N, 3, 1, 180)`

Same Conv2d architecture as nb07c/nb08, but with **kernel `(1, 3)`** and **pool `(1, 2)`** so convolution only operates along the longitude axis (the singleton lat axis is preserved through the network). After two pools: `180 → 90 → 45`. Then AdaptiveAvgPool → 32-dim feature → 2-dim FC.

In [ ]:
class MJOEncoderNoL2(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        # Convolutions act only along longitude: kernel (1,3), padding (0,1)
        self.conv1 = nn.Conv2d(3,  16, kernel_size=(1, 3), padding=(0, 1), bias=False)
        self.bn1   = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d((1, 2))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(1, 3), padding=(0, 1), bias=False)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d((1, 2))
        self.conv3 = nn.Conv2d(32, 32, kernel_size=(1, 3), padding=(0, 1), bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(32, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).view(x.size(0), -1)
        return self.fc(x)   # no L2 normalization


def InfoNCE_loss_raw(z_A, z_B, temperature):
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature
    labels_ = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels_)


def vicreg_terms(z, gamma=1.0, eps=1e-4):
    # VICReg variance+covariance (nb07d collapse fix): variance hinge + decorrelation
    z = z - z.mean(0, keepdim=True)
    std = torch.sqrt(z.var(0) + eps)
    var_loss = torch.mean(F.relu(gamma - std))
    B, D = z.shape
    cov = (z.T @ z) / (B - 1)
    off = cov - torch.diag(torch.diag(cov))
    cov_loss = off.pow(2).sum() / D
    return var_loss, cov_loss


# Sanity check
enc_test = MJOEncoderNoL2(embedding_dim=EMBEDDING_DIM)
total_params = sum(p.numel() for p in enc_test.parameters())
dummy = torch.randn(4, 3, 1, 180)
out   = enc_test(dummy)
print(f'Total parameters: {total_params:,}')
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}  (expect [4, {EMBEDDING_DIM}])')
print(f'Init norms: {out.norm(dim=1).detach().numpy()}')

## Cell 5 — Initialize Model + Optimizer

In [ ]:
encoder   = MJOEncoderNoL2(embedding_dim=EMBEDDING_DIM).to(device)
optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f'Model on:    {device}')
print(f'Parameters:  {sum(p.numel() for p in encoder.parameters()):,}')
print(f'\nTraining config:')
print(f'  Epochs:        {EPOCHS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Temperature:   {TEMPERATURE}')
print(f'  LR:            {LR} → 1e-5 (cosine)')
print(f'  Weight decay:  {WEIGHT_DECAY}')
print(f'  Train batches/epoch: {len(train_loader)}')

## Cell 6 — Training Loop (with Norm Trajectory Tracking)

In [ ]:
from tqdm.notebook import tqdm

history = {'train_loss': [], 'val_loss': [], 'epoch_time': [],
           'mean_norm': [], 'std_norm': [], 'max_norm': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    encoder.train()
    train_loss = 0.0
    epoch_norms = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for fA, fB in pbar:
        fA = fA.to(device, non_blocking=True)
        fB = fB.to(device, non_blocking=True)
        zA = encoder(fA); zB = encoder(fB)
        with torch.no_grad():
            epoch_norms.extend(zA.norm(dim=1).cpu().numpy().tolist())
            epoch_norms.extend(zB.norm(dim=1).cpu().numpy().tolist())
        loss = InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE)
        if USE_VICREG:
            vA, cA = vicreg_terms(zA); vB, cB = vicreg_terms(zB)
            loss = loss + VICREG_VAR * (vA + vB) / 2 + VICREG_COV * (cA + cB) / 2
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    train_loss /= len(train_loader)

    encoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for fA, fB in val_loader:
            fA = fA.to(device, non_blocking=True); fB = fB.to(device, non_blocking=True)
            zA = encoder(fA); zB = encoder(fB)
            val_loss += InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE).item()
    val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else float('nan')
    scheduler.step()

    en = np.array(epoch_norms); et = time.time() - t0
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['epoch_time'].append(et)
    history['mean_norm'].append(float(en.mean())); history['std_norm'].append(float(en.std()))
    history['max_norm'].append(float(en.max()))
    print(f'ep {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'norm μ={en.mean():.3f} σ={en.std():.3f} max={en.max():.2f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if (epoch + 1) % 10 == 0:
        torch.save({'epoch': epoch+1, 'model_state_dict': encoder.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss, 'val_loss': val_loss},
                   f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_epoch_{epoch+1}.pth')

torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_final.pth')
with open(f'{CHECKPOINT_DIR}/training_history_{RUN_TAG}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Total time: {sum(history["epoch_time"])/60:.1f} min')
print(f'Best val loss: {min(history["val_loss"]):.4f}  (epoch {history["val_loss"].index(min(history["val_loss"]))+1})')
print(f'Final mean norm: {history["mean_norm"][-1]:.3f}  max norm: {history["max_norm"][-1]:.2f}')

## Cell 7 — Training Curves + Norm Trajectory

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0].axhline(np.log(64), color='gray', linestyle='--', alpha=0.5, label='log(64)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title('Training Curves', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['mean_norm'], label='Mean norm', linewidth=2)
axes[1].fill_between(range(len(history['mean_norm'])),
                     np.array(history['mean_norm']) - np.array(history['std_norm']),
                     np.array(history['mean_norm']) + np.array(history['std_norm']),
                     alpha=0.3, label='±1σ')
axes[1].plot(history['max_norm'], label='Max norm', color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Embedding norm')
axes[1].set_title('Norm Trajectory', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['epoch_time'], color='green', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (s)')
axes[2].set_title('Epoch Time', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

if max(history['max_norm']) > 100:
    print('\n⚠️  Norm explosion. Consider increasing WEIGHT_DECAY to 1e-3.')
else:
    print(f'\n✓ Norms stable (max ever = {max(history["max_norm"]):.2f})')

## Cell 8 — Extract Embeddings + 4-Panel Scatter

In [ ]:
encoder.eval()
embeddings_2d = np.zeros((len(X), EMBEDDING_DIM), dtype=np.float32)
with torch.no_grad():
    for start in range(0, len(X), 128):
        end = min(start + 128, len(X))
        batch = torch.from_numpy(X[start:end]).float().to(device)
        embeddings_2d[start:end] = encoder(batch).cpu().numpy()

np.save(f'{RESULTS_DIR}/embeddings.npy', embeddings_2d)

norms  = np.linalg.norm(embeddings_2d, axis=1)
angles = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
print(f'Embeddings shape: {embeddings_2d.shape}')
print(f'Norm:  min={norms.min():.3f} max={norms.max():.3f} mean={norms.mean():.3f} std={norms.std():.3f}')
print(f'Angle: spread={angles.max()-angles.min():.3f} rad')

# 4-panel scatter on val set (active MJO only for clarity)
lv = labels.loc[val_idx]
act_v = ~lv['weak_mjo'].values
Z_val = embeddings_2d[val_idx][act_v]
lv_a  = lv[act_v]

phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}

rng_lo = min(Z_val.min(), -1.1); rng_hi = max(Z_val.max(), 1.1)
pad = 0.1 * (rng_hi - rng_lo); rng_lo -= pad; rng_hi += pad

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# (a) by RMM phase
ax = axes[0]
for ph in range(1, 9):
    m = lv_a['phase'] == ph
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[phase_colors[ph-1]], s=12, alpha=0.7, label=f'P{ph}')
ax.set_title('By RMM Phase', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)

# (b) by ENSO
ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = lv_a['enso_category'] == cat
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=enso_palette[cat], marker=enso_marker[cat], s=12, alpha=0.5, label=cat)
ax.set_title('By ENSO', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(); ax.grid(alpha=0.2)

# (c) by RMM amplitude
ax = axes[2]
sc = ax.scatter(Z_val[:, 0], Z_val[:, 1], c=lv_a['amplitude'].values,
                cmap='viridis', s=12, alpha=0.7)
ax.set_title('By RMM Amplitude', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
plt.colorbar(sc, ax=ax, label='RMM amplitude'); ax.grid(alpha=0.2)

# (d) angular histogram
ax = axes[3]
angles_val = np.arctan2(Z_val[:, 1], Z_val[:, 0])
ax.hist(angles_val, bins=36, color='steelblue', alpha=0.7)
ax.set_xlabel('Angle θ (rad)'); ax.set_ylabel('Count')
ax.set_title('Angular Distribution (val, active MJO)', fontweight='bold')
ax.set_xlim(-np.pi, np.pi); ax.grid(alpha=0.3)

plt.suptitle(f'MJO Supervised 2D — Val Embeddings (active MJO, n={len(Z_val)})',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/embedding_2d_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/embedding_2d_overview.png')

## Cell 8b — ENSO Marginal Histograms (El Nino vs La Nina)

Marginal-histogram companion to the ENSO scatter above ([r-bloggers example 8.41](https://www.r-bloggers.com/2011/06/example-8-41-scatterplot-with-marginal-histograms/)):
one density-normalised histogram per ENSO category on each latent axis, so the
El Nino and La Nina **means** can be compared directly instead of eyeballed from
an overplotted cloud. Neutral days are kept as grey background context; all
statistics are El Nino vs La Nina only.

**Three figures:**

1. `enso_marginal_hist.png` — **raw** marginals, pooled over all phases. Answers
   *is the latent cloud as a whole shifted?* This mixes two effects: (i) within a
   phase, EN days sit somewhere different from LN days, and (ii) EN and LN years
   simply spend different amounts of time in each phase.
2. `enso_marginal_residual.png` — same layout after **removing the per-phase
   mean**, so a difference cannot come from unequal phase occupancy. This isolates
   the within-phase displacement — the quantity the ENSO displacement z-score
   measures. The two can disagree: the raw shift can be ~0 while the per-phase
   displacement is large, because displacements at opposite points of the ring
   cancel when pooled.
3. `enso_marginal_polar.png` — the ring's natural coordinates: radius
   (oscillation amplitude) and angle (phase clock).

**Reading the numbers.** `enso_category` is a per-year JJA Nino-3.4 label and the
fields decorrelate in ~15-25 d, so the independent unit is the **year**, not the day.
The day-level t-test is printed for reference but is over-confident by roughly
sqrt(n_days / n_years); trust the **year-block bootstrap 95% CI** and the year-label
permutation p instead.

`SCOPE = 'all'` (default) uses every day, matching the ENSO-displacement cell these
figures visualise. `SCOPE = 'val'` restricts to held-out years, but the val split is
every 5th year, leaving only ~2-3 El Nino and ~2-3 La Nina years — too few for the
year-level test to resolve anything (the permutation p then has a floor of
1/C(n_EN+n_LN, n_EN)). The cell prints that floor so the limit is visible.


In [ ]:
# ── ENSO marginal histograms — El Nino vs La Nina  (Session 61) ──────────────
# Pure plotting cell: no GPU, no X, no model, no retraining.
TAG         = 'mjo_sup'
TITLE       = 'MJO Supervised 2D (active MJO)'
PHASE_COL   = 'phase'
UNIT_CIRCLE = False   # no L2 norm -> radius tracks RMM amplitude (r = 0.50, Session 24)
SCOPE       = 'all'   # 'all' = every day, matching the ENSO-displacement cell below.
                      # 'val' = held-out years only, but that leaves ~2-3 EN and LN
                      # YEARS, and the year is the independent unit here, so the
                      # year-block CI becomes uninformative. See the printout.
N_BOOT      = 2000
PERSIST_DIR = f'{PROJECT_DIR}/results/enso_marginal'

# NOTE: this encoder was trained on ENSO-conditioned pairs, so with SCOPE='all'
# the training years are in-sample and any separation is partly fitted. The SSL
# notebooks (08 / 15) have no such contamination — that is the fair comparison.

import os
import numpy as np
import pandas as pd

try:                                    # 1. variables still in memory
    _Zf, _lf = embeddings_2d, labels
    print('Using the in-memory embeddings.')
except NameError:
    _p = f'{RESULTS_DIR}/embeddings.npy'
    if os.path.exists(_p):              # 2. reload what this notebook saved
        _Zf = np.load(_p)
        _lf = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])
        print(f'Reloaded embeddings.npy ({len(_Zf)} rows).')
    else:                               # 3. RESULTS_DIR was cleared -> this cell's cache
        _c  = np.load(f'{PERSIST_DIR}/{TAG}_marginal_inputs.npz')
        _Zf = _c['Z']
        _lf = pd.DataFrame({PHASE_COL: _c['phase'], 'enso_category': _c['cat'],
                            'date': pd.to_datetime(_c['year'].astype(str))})  # only .dt.year is used
        SCOPE = 'cache'
        print('Restored from the cached npz written by an earlier run of this cell.')

assert len(_Zf) == len(_lf), f'embeddings {len(_Zf)} vs labels {len(_lf)} mismatch'
if SCOPE == 'val':                      # same split rule as the training cells above
    _m = _lf['date'].dt.year.isin(sorted(_lf['date'].dt.year.unique())[::5]).values
else:
    _m = np.ones(len(_lf), dtype=bool)
if 'weak_mjo' in _lf.columns:   # absent from the tier-3 cache, which is already filtered
    _m &= (~_lf['weak_mjo'].values) & _lf['phase'].between(1, 8).values
Z_m, lab_m = _Zf[_m], _lf.loc[_m]
print(f'Scope: {SCOPE}  ->  {int(_m.sum())} of {len(_lf)} days')

# ── shared body (identical in notebooks 07 / 08 / 14 / 15) ───────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats

os.makedirs(PERSIST_DIR, exist_ok=True)

Z_m   = np.asarray(Z_m, dtype=float)
lab_m = lab_m.reset_index(drop=True)
keep  = lab_m[PHASE_COL].between(1, 8).values & np.isfinite(Z_m).all(axis=1)
Z_m, lab_m = Z_m[keep], lab_m.loc[keep].reset_index(drop=True)

cat_m   = lab_m['enso_category'].values
phase_m = lab_m[PHASE_COL].values.astype(int)
year_m  = lab_m['date'].dt.year.values
is_en, is_ln, is_nu = cat_m == 'El Nino', cat_m == 'La Nina', cat_m == 'Neutral'

# Cache inputs so this figure can be regenerated even if RESULTS_DIR is wiped
np.savez(f'{PERSIST_DIR}/{TAG}_marginal_inputs.npz',
         Z=Z_m, phase=phase_m, year=year_m, cat=cat_m.astype('U10'))

from math import comb
_ny_en, _ny_ln = len(np.unique(year_m[is_en])), len(np.unique(year_m[is_ln]))
_p_floor = 1.0 / comb(_ny_en + _ny_ln, _ny_en)
print(f'{TITLE}:  n={len(Z_m)} days  '
      f'(EN {is_en.sum()}, LN {is_ln.sum()}, Neutral {is_nu.sum()})')
print(f'Independent units: {_ny_en} El Nino years, {_ny_ln} La Nina years  '
      f'-> year-permutation p cannot go below {_p_floor:.3f}')
if _p_floor > 0.02:
    print('  WARNING: too few years for the year-level test to resolve significance. '
          "Use SCOPE='all' (or read the bootstrap CI, not the p-value).")
print(f'Years: EN {sorted(np.unique(year_m[is_en]))}')
print(f'       LN {sorted(np.unique(year_m[is_ln]))}')

# ── circular helpers ────────────────────────────────────────────────────────
def _cmean(a):
    return np.arctan2(np.sin(a).mean(), np.cos(a).mean())

def _wrap(a):
    return np.arctan2(np.sin(a), np.cos(a))

def _mean(v, circular):
    return _cmean(v) if circular else v.mean()

def _delta(ve, vl, circular):
    """EN − LN mean difference; smallest signed arc if circular."""
    return _wrap(_cmean(ve) - _cmean(vl)) if circular else ve.mean() - vl.mean()

# ── statistics ──────────────────────────────────────────────────────────────
# ENSO category is a per-year label and the fields decorrelate in ~15-25 d, so
# the independent unit is the YEAR, not the day. Day-level p-values are shown
# for reference only; the year-block bootstrap CI is the honest uncertainty.
def _stats(v, circular, seed=0):
    ve, vl = v[is_en], v[is_ln]
    yr_en, yr_ln = np.unique(year_m[is_en]), np.unique(year_m[is_ln])
    obs = _delta(ve, vl, circular)

    if circular:
        d_pool = np.concatenate([_wrap(ve - _cmean(ve)), _wrap(vl - _cmean(vl))])
        sd_p, cohen = d_pool.std(ddof=1), np.nan
        rng = np.random.default_rng(seed + 7)
        pool = np.concatenate([ve, vl])
        lab_p = np.r_[np.ones(len(ve), bool), np.zeros(len(vl), bool)]
        null = np.empty(1000)
        for i in range(1000):
            s = rng.permutation(lab_p)
            null[i] = abs(_delta(pool[s], pool[~s], True))
        p_naive = float((null >= abs(obs)).mean())
    else:
        sd_p = np.sqrt(((len(ve)-1)*ve.var(ddof=1) + (len(vl)-1)*vl.var(ddof=1))
                       / (len(ve) + len(vl) - 2))
        cohen  = obs / (sd_p + 1e-12)
        p_naive = float(stats.ttest_ind(ve, vl, equal_var=False).pvalue)

    by_year = {y: np.where(year_m == y)[0] for y in np.unique(year_m)}
    rng = np.random.default_rng(seed)

    boot = np.empty(N_BOOT)
    for i in range(N_BOOT):
        se = rng.choice(yr_en, len(yr_en), replace=True)
        sl = rng.choice(yr_ln, len(yr_ln), replace=True)
        boot[i] = _delta(np.concatenate([v[by_year[y]] for y in se]),
                         np.concatenate([v[by_year[y]] for y in sl]), circular)
    if circular:   # percentiles of raw angles wrap; centre on the observed value
        lo, hi = obs + np.percentile(_wrap(boot - obs), [2.5, 97.5])
    else:
        lo, hi = np.percentile(boot, [2.5, 97.5])

    all_yr, k = np.concatenate([yr_en, yr_ln]), len(yr_en)
    null = np.empty(N_BOOT)
    for i in range(N_BOOT):
        s  = rng.permutation(all_yr)
        ie = np.concatenate([by_year[y] for y in s[:k]])
        il = np.concatenate([by_year[y] for y in s[k:]])
        null[i] = _delta(v[ie], v[il], circular)
    p_perm = float((np.abs(null) >= abs(obs)).mean())

    R_en = np.hypot(np.cos(ve).mean(), np.sin(ve).mean()) if circular else np.nan
    R_ln = np.hypot(np.cos(vl).mean(), np.sin(vl).mean()) if circular else np.nan
    return dict(mean_EN=_mean(ve, circular), mean_LN=_mean(vl, circular),
                R_EN=R_en, R_LN=R_ln,
                delta=obs, cohens_d=cohen, sd_pooled=sd_p,
                n_EN_days=int(is_en.sum()), n_LN_days=int(is_ln.sum()),
                n_EN_years=len(yr_en), n_LN_years=len(yr_ln),
                p_naive_day=p_naive, boot_lo=lo, boot_hi=hi, p_perm_year=p_perm)

def _residual(v, circular):
    """Remove the per-phase mean so a difference cannot come from unequal
    phase occupancy between EN and LN years."""
    out = v.astype(float).copy()
    for p in np.unique(phase_m):
        m = phase_m == p
        out[m] = _wrap(v[m] - _cmean(v[m])) if circular else v[m] - v[m].mean()
    return out

# ── figures ─────────────────────────────────────────────────────────────────
EN_C, LN_C, NU_C = '#d62728', '#1f77b4', '#7f7f7f'

def _marginal(ax, v, horiz, bins, circular=False):
    kw = dict(bins=bins, density=True, orientation='horizontal' if horiz else 'vertical')
    ax.hist(v[is_nu], histtype='step', color=NU_C, lw=0.8, alpha=0.55, **kw)
    for m, c, n in [(is_ln, LN_C, 'La Nina'), (is_en, EN_C, 'El Nino')]:
        ax.hist(v[m], histtype='stepfilled', color=c, alpha=0.40, label=n, **kw)
        ax.hist(v[m], histtype='step', color=c, lw=1.4, **kw)
        mu = _cmean(v[m]) if circular else v[m].mean()
        (ax.axhline if horiz else ax.axvline)(mu, color=c, ls='--', lw=1.6)

def _joint(Z, s1, s2, title, fname, xl, yl):
    fig = plt.figure(figsize=(9.5, 9.5))
    gs  = GridSpec(2, 2, width_ratios=[4, 1.15], height_ratios=[1.15, 4],
                   hspace=0.06, wspace=0.06, figure=fig)
    axm = fig.add_subplot(gs[1, 0])
    axt = fig.add_subplot(gs[0, 0], sharex=axm)
    axr = fig.add_subplot(gs[1, 1], sharey=axm)
    axs = fig.add_subplot(gs[0, 1]); axs.axis('off')

    axm.scatter(Z[is_nu, 0], Z[is_nu, 1], c=NU_C, marker='o', s=14, alpha=0.12,
                label=f'Neutral (n={is_nu.sum()})', zorder=1)
    axm.scatter(Z[is_ln, 0], Z[is_ln, 1], c=LN_C, marker='s', s=20, alpha=0.60,
                label=f'La Nina (n={is_ln.sum()})', zorder=2)
    axm.scatter(Z[is_en, 0], Z[is_en, 1], c=EN_C, marker='^', s=22, alpha=0.60,
                label=f'El Nino (n={is_en.sum()})', zorder=3)
    for m, c in [(is_ln, LN_C), (is_en, EN_C)]:
        axm.plot(Z[m, 0].mean(), Z[m, 1].mean(), marker='X', ms=17, mfc=c,
                 mec='k', mew=1.6, ls='none', zorder=4)
    axm.axhline(0, color='k', lw=0.4, alpha=0.3); axm.axvline(0, color='k', lw=0.4, alpha=0.3)
    axm.set_xlabel(xl); axm.set_ylabel(yl); axm.set_aspect('equal')
    axm.grid(alpha=0.2); axm.legend(loc='upper left', fontsize=9, framealpha=0.9)

    b1 = np.histogram_bin_edges(Z[:, 0], bins=30)
    b2 = np.histogram_bin_edges(Z[:, 1], bins=30)
    _marginal(axt, Z[:, 0], False, b1); _marginal(axr, Z[:, 1], True, b2)
    axt.set_ylabel('density'); axr.set_xlabel('density')
    plt.setp(axt.get_xticklabels(), visible=False)
    plt.setp(axr.get_yticklabels(), visible=False)
    axt.grid(alpha=0.2); axr.grid(alpha=0.2)
    axt.set_title(title, fontweight='bold', fontsize=12, pad=10)

    def _line(n, s):
        star = '*' if (s['boot_lo'] > 0) or (s['boot_hi'] < 0) else ' '
        return (f"{n}:  EN {s['mean_EN']:+.3f}   LN {s['mean_LN']:+.3f}\n"
                f"   delta {s['delta']:+.3f}{star}  d={s['cohens_d']:+.2f}\n"
                f"   year-block 95% CI [{s['boot_lo']:+.3f}, {s['boot_hi']:+.3f}]\n"
                f"   p_perm(year)={s['p_perm_year']:.3f}   "
                f"p_t(day)={s['p_naive_day']:.1e}")
    axs.text(0, 1, _line(xl, s1) + '\n\n' + _line(yl, s2) +
             f"\n\nyears: EN {s1['n_EN_years']}  LN {s1['n_LN_years']}"
             "\n* = CI excludes 0",
             transform=axs.transAxes, va='top', ha='left', fontsize=7.4,
             family='monospace',
             bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.95))
    plt.savefig(f'{RESULTS_DIR}/{fname}', dpi=150, bbox_inches='tight')
    plt.savefig(f'{PERSIST_DIR}/{TAG}_{fname}', dpi=150, bbox_inches='tight')
    plt.show()

rows = {}
rows['z1_raw'] = _stats(Z_m[:, 0], False)
rows['z2_raw'] = _stats(Z_m[:, 1], False)
_joint(Z_m, rows['z1_raw'], rows['z2_raw'],
       f'{TITLE} — ENSO marginals (raw)\n'
       'pooled over all phases: mixes within-phase displacement with phase occupancy',
       'enso_marginal_hist.png', 'z1', 'z2')

Z_res = np.column_stack([_residual(Z_m[:, 0], False), _residual(Z_m[:, 1], False)])
rows['z1_resid'] = _stats(Z_res[:, 0], False)
rows['z2_resid'] = _stats(Z_res[:, 1], False)
_joint(Z_res, rows['z1_resid'], rows['z2_resid'],
       f'{TITLE} — ENSO marginals (per-phase mean removed)\n'
       'within-phase displacement only — the quantity the displacement z-score measures',
       'enso_marginal_residual.png', 'z1 residual', 'z2 residual')

# ── radius / angle marginals (the ring's natural coordinates) ───────────────
r_m, th_m = np.hypot(Z_m[:, 0], Z_m[:, 1]), np.arctan2(Z_m[:, 1], Z_m[:, 0])
rows['r_raw']      = _stats(r_m, False)
rows['theta_raw']  = _stats(th_m, True)
rows['r_resid']    = _stats(_residual(r_m, False), False)
rows['theta_resid']= _stats(_residual(th_m, True), True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
_marginal(axes[0], r_m, False, np.histogram_bin_edges(r_m, bins=30))
rt = 'radius (== 1 by construction: L2-normalised encoder — uninformative)' if UNIT_CIRCLE \
     else 'radius  (oscillation amplitude)'
axes[0].set_title(rt, fontweight='bold', color='red' if UNIT_CIRCLE else 'black', fontsize=11)
axes[0].set_xlabel('|z|'); axes[0].set_ylabel('density')
axes[0].legend(); axes[0].grid(alpha=0.25)
s = rows['r_raw']
axes[0].annotate(f"delta={s['delta']:+.3f}  CI [{s['boot_lo']:+.3f}, {s['boot_hi']:+.3f}]",
                 xy=(0.02, 0.92), xycoords='axes fraction', fontsize=9, family='monospace')

_marginal(axes[1], th_m, False, np.linspace(-np.pi, np.pi, 37), circular=True)
axes[1].set_title('angle  (phase clock)', fontweight='bold', fontsize=11)
axes[1].set_xlabel('theta (rad)'); axes[1].set_ylabel('density')
axes[1].set_xlim(-np.pi, np.pi); axes[1].legend(); axes[1].grid(alpha=0.25)
s = rows['theta_raw']
axes[1].annotate(f"circ delta={s['delta']:+.3f} rad  CI [{s['boot_lo']:+.3f}, {s['boot_hi']:+.3f}]",
                 xy=(0.02, 0.92), xycoords='axes fraction', fontsize=9, family='monospace')
plt.suptitle(f'{TITLE} — ENSO marginals in ring coordinates', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_marginal_polar.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{PERSIST_DIR}/{TAG}_enso_marginal_polar.png', dpi=150, bbox_inches='tight')
plt.show()

df_marg = pd.DataFrame(rows).T
df_marg.index.name = 'axis'
df_marg.insert(0, 'representation', TAG)
df_marg.to_csv(f'{RESULTS_DIR}/enso_marginal_stats.csv')
df_marg.to_csv(f'{PERSIST_DIR}/{TAG}_enso_marginal_stats.csv')

print('\nEN - LN mean difference (delta), with year-block 95% CI:')
print(df_marg[['mean_EN', 'mean_LN', 'delta', 'cohens_d',
               'boot_lo', 'boot_hi', 'p_perm_year', 'p_naive_day']].round(4).to_string())
_R = rows['theta_raw']
if min(_R['R_EN'], _R['R_LN']) < 0.10:
    print(f"\nNOTE: angular resultant length is small (EN {_R['R_EN']:.3f}, "
          f"LN {_R['R_LN']:.3f}) — the angles are near-uniform around the ring, so the "
          "circular MEAN of theta is ill-defined. Read theta_resid, not theta_raw.")
sig = df_marg[(df_marg.boot_lo > 0) | (df_marg.boot_hi < 0)].index.tolist()
print(f'\nAxes whose year-block CI excludes 0: {sig if sig else "none"}')
print('Reminder: p_naive_day treats each day as independent and is over-confident '
      'by ~sqrt(n_days/n_years); read the CI and p_perm_year instead.')
print(f'\nSaved to {RESULTS_DIR}/ and {PERSIST_DIR}/: '
      'enso_marginal_hist.png, enso_marginal_residual.png, '
      'enso_marginal_polar.png, enso_marginal_stats.csv')


## Cell 9 — Radius Diagnostics

Does the freed radius encode RMM amplitude (or anything systematic)?

In [ ]:
from scipy.stats import pearsonr, spearmanr, f_oneway

active_mask = (~labels['weak_mjo'].values) & (labels['phase'].between(1, 8).values)
radii_a  = norms[active_mask]
ampl_a   = labels.loc[active_mask, 'amplitude'].values
phase_a  = labels.loc[active_mask, 'phase'].values
enso_a   = labels.loc[active_mask, 'enso_category'].values

r_p, p_p = pearsonr(radii_a, ampl_a)
r_s, p_s = spearmanr(radii_a, ampl_a)
print(f'Radius vs RMM amplitude (active MJO only):')
print(f'  Pearson r  = {r_p:.3f}  p = {p_p:.2e}')
print(f'  Spearman r = {r_s:.3f}  p = {p_s:.2e}')

radii_by_phase = [radii_a[phase_a == ph] for ph in range(1, 9)]
f_ph, p_ph = f_oneway(*radii_by_phase)
print(f'\nRadius by RMM phase (ANOVA): F={f_ph:.2f}  p={p_ph:.2e}')

radii_by_enso = {c: radii_a[enso_a == c] for c in ['El Nino', 'Neutral', 'La Nina']}
f_en, p_en = f_oneway(*radii_by_enso.values())
print(f'Radius by ENSO (ANOVA):      F={f_en:.2f}  p={p_en:.2e}')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ax = axes[0]
ax.scatter(ampl_a, radii_a, alpha=0.2, s=6)
ax.set_xlabel('RMM amplitude'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius vs Amplitude\nPearson r={r_p:.3f}', fontweight='bold')
z_fit = np.polyfit(ampl_a, radii_a, 1)
x_fit = np.linspace(ampl_a.min(), ampl_a.max(), 100)
ax.plot(x_fit, np.polyval(z_fit, x_fit), 'r-', linewidth=2)
ax.grid(alpha=0.3)

ax = axes[1]
bp = ax.boxplot(radii_by_phase, positions=range(1, 9), widths=0.6,
                patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], plt.cm.tab10(np.linspace(0, 0.8, 8))):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_xlabel('RMM phase'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by Phase\nANOVA F={f_ph:.2f}', fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[2]
cats = ['El Nino', 'Neutral', 'La Nina']
bp = ax.boxplot([radii_by_enso[c] for c in cats], positions=[1, 2, 3], widths=0.6,
                patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], cats):
    patch.set_facecolor({'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}[c])
    patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(cats)
ax.set_xlabel('ENSO category'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by ENSO\nANOVA F={f_en:.2f}', fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/radius_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

radius_summary = {
    'radius_vs_amplitude_pearson':  [float(r_p), float(p_p)],
    'radius_vs_amplitude_spearman': [float(r_s), float(p_s)],
    'radius_by_phase_anova': [float(f_ph), float(p_ph)],
    'radius_by_enso_anova':  [float(f_en), float(p_en)],
    'mean_radius': float(radii_a.mean()),
    'std_radius':  float(radii_a.std()),
}
with open(f'{RESULTS_DIR}/radius_summary.json', 'w') as f:
    json.dump(radius_summary, f, indent=2)
print(f'Saved: {RESULTS_DIR}/radius_summary.json')

## Cell 10 — Linear Probes (RMM Phase + ENSO Balanced)

Random baselines: 12.5% (8-phase) and 33.3% (3-class balanced).  
Probe only over **active MJO days** (consistent with how training pairs were defined).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GroupKFold

active_idx = np.where(active_mask)[0]
train_act  = np.intersect1d(train_idx, active_idx)
val_act    = np.intersect1d(val_idx,   active_idx)

Z_train = embeddings_2d[train_act]
Z_val_  = embeddings_2d[val_act]
gkf = GroupKFold(n_splits=5)
year_groups_act = labels.loc[active_idx, 'date'].dt.year.values

# RMM phase
y_tr = labels.loc[train_act, 'phase'].values
y_va = labels.loc[val_act,   'phase'].values
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(Z_train, y_tr)
phase_val = float(accuracy_score(y_va, clf.predict(Z_val_)))
cv_phase = cross_val_score(clf, embeddings_2d[active_idx], labels.loc[active_idx, 'phase'].values,
                            cv=gkf, groups=year_groups_act, scoring='accuracy', n_jobs=-1)

# ENSO balanced
y_tr_e = labels.loc[train_act, 'enso_category'].values
y_va_e = labels.loc[val_act,   'enso_category'].values
clf_b = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
clf_b.fit(Z_train, y_tr_e)
enso_bal = float(balanced_accuracy_score(y_va_e, clf_b.predict(Z_val_)))
cv_enso = cross_val_score(clf_b, embeddings_2d[active_idx], labels.loc[active_idx, 'enso_category'].values,
                           cv=gkf, groups=year_groups_act, scoring='balanced_accuracy', n_jobs=-1)

probe_results = {
    'RMM Phase': {'val_acc': phase_val,  'cv_mean': float(cv_phase.mean()), 'cv_std': float(cv_phase.std())},
    'ENSO bal':  {'val_acc': enso_bal,   'cv_mean': float(cv_enso.mean()),  'cv_std': float(cv_enso.std())},
}

print('=' * 70)
print('MJO SUPERVISED — LINEAR PROBE RESULTS')
print('=' * 70)
print(f'RMM phase val acc:    {phase_val*100:.1f}%  (random 12.5%)')
print(f'RMM phase 5-fold CV:  {cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%')
print(f'ENSO  bal-acc val:    {enso_bal*100:.1f}%   (random 33.3%)')
print(f'ENSO  bal-acc CV:     {cv_enso.mean()*100:.1f}% ± {cv_enso.std()*100:.1f}%')
print(f'\nClassification report (RMM phase val):')
print(classification_report(y_va, clf.predict(Z_val_), zero_division=0))

with open(f'{RESULTS_DIR}/linear_probe_results.json', 'w') as f:
    json.dump(probe_results, f, indent=2)

## Cell 11 — ENSO Displacement Z-Score

Same statistic as BSISO nb07c. Computed over active MJO days only.

In [ ]:
labels_act = labels.loc[active_mask].reset_index(drop=True)
emb_act = embeddings_2d[active_mask]

phases = range(1, 9)
disp_mag = []
for ph in phases:
    mEN = (labels_act['phase'] == ph) & (labels_act['enso_category'] == 'El Nino')
    mLN = (labels_act['phase'] == ph) & (labels_act['enso_category'] == 'La Nina')
    if mEN.sum() < 3 or mLN.sum() < 3:
        disp_mag.append(np.nan); continue
    cEN = emb_act[mEN].mean(axis=0); cLN = emb_act[mLN].mean(axis=0)
    disp_mag.append(np.linalg.norm(cEN - cLN))

rng = np.random.default_rng(42)
baseline_mag = []
for _ in range(100):
    shuf = labels_act['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
    mtrl = []
    for ph in phases:
        mph = (labels_act['phase'] == ph).values
        mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        mtrl.append(np.linalg.norm(emb_act[mEN].mean(axis=0) - emb_act[mLN].mean(axis=0)))
    if mtrl: baseline_mag.append(np.mean(mtrl))

bmu = float(np.mean(baseline_mag)); bsd = float(np.std(baseline_mag))
obs_mu = float(np.nanmean(disp_mag))
z_score_sup = float((obs_mu - bmu) / (bsd + 1e-8))

print(f'EN−LN displacement summary (MJO supervised):')
print(f'  Observed mean: {obs_mu:.4f}')
print(f'  Null baseline: {bmu:.4f} ± {bsd:.4f}')
print(f'  Z-score:       {z_score_sup:.2f}')
print(f'  (BSISO supervised 2D baseline: z=2.53;  SSL: z=14.55)')

fig, ax = plt.subplots(figsize=(8, 5))
valid_p = [p for p, m in zip(phases, disp_mag) if not np.isnan(m)]
valid_m = [m for m in disp_mag if not np.isnan(m)]
ax.bar(valid_p, valid_m, color='steelblue', alpha=0.8)
ax.axhline(bmu, color='red', linestyle='--', linewidth=1.5, label=f'Null mean ({bmu:.3f})')
ax.axhline(bmu + 2*bsd, color='red', linestyle=':', linewidth=1, label='Null +2σ')
ax.axhline(obs_mu, color='steelblue', linewidth=2, label=f'Observed mean ({obs_mu:.3f})')
ax.set_xticks(range(1, 9))
ax.set_xlabel('RMM Phase'); ax.set_ylabel('||EN−LN||')
ax.set_title(f'MJO Supervised ENSO Displacement, z = {z_score_sup:.2f}', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 — Summary + Auto-Decision

In [ ]:
import time as _t

# BSISO reference numbers for context
BSISO_SUP_2D = {'phase_val': 0.583, 'phase_cv': '65.7% ± 4.1%', 'enso_bal': 0.346, 'z': 2.53}

rows = [
    ['BSISO 2D supervised (nb07c)', f"{BSISO_SUP_2D['phase_val']*100:.1f}%", BSISO_SUP_2D['phase_cv'],
     f"{BSISO_SUP_2D['enso_bal']*100:.1f}%", f"{BSISO_SUP_2D['z']:.2f}"],
    ['**MJO 2D supervised (this nb)**', f'{phase_val*100:.1f}%',
     f'{cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%',
     f'{enso_bal*100:.1f}%', f'{z_score_sup:.2f}'],
]
df_comp = pd.DataFrame(rows, columns=['Configuration', 'Phase val', '5-fold CV', 'ENSO bal-acc', 'z-score'])

print('=' * 100)
print('MJO SUPERVISED 2D SUMMARY')
print('=' * 100)
print(df_comp.to_string(index=False))

# Auto-decision
phase_pass = phase_val >= 0.35   # 8-class, expect ≥ 35% for a meaningful signal (vs 12.5% random)
z_pass     = z_score_sup >= 2.0
norm_ok    = max(history['max_norm']) < 100

if not norm_ok:
    decision = 'NORM_EXPLOSION'
    decision_text = f"Norms exploded (max {max(history['max_norm']):.1f}). Increase WEIGHT_DECAY to 1e-3."
elif phase_pass and z_pass:
    decision = 'BASELINE_ESTABLISHED'
    decision_text = (f"**Supervised baseline established.** Phase val {phase_val*100:.1f}%, z={z_score_sup:.2f}. "
                     f"Use as the supervised reference for the three-way comparison (nb16). "
                     f"Proceed to nb15 (SSL temporal).")
elif phase_pass:
    decision = 'PHASE_OK_ENSO_WEAK'
    decision_text = (f"Phase signal recovered ({phase_val*100:.1f}%) but ENSO modulation weak (z={z_score_sup:.2f}). "
                     f"Still usable as supervised baseline.")
else:
    decision = 'WEAK_SIGNAL'
    decision_text = (f"Weak supervised signal (phase {phase_val*100:.1f}%, z={z_score_sup:.2f}). "
                     f"Inspect training curves and verify input pipeline before nb15.")

print('\n' + '=' * 70)
print(f'DECISION: {decision}')
print('=' * 70)
print(decision_text)

summary = f"""# MJO Supervised 2D Summary

**Auto-generated by notebook 14.**  
**Date:** {_t.strftime('%Y-%m-%d')}  
**Model:** `encoder_{RUN_TAG}_final.pth`, no L2 norm, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}  
**Input:** `X_MJO.npy` shape (N, 3, 1, 180), channels [u850, OLR, u200]  
**Pairs:** RMM phase + ENSO category (active MJO only, amplitude ≥ 1)

## Headline results

{df_comp.to_markdown(index=False)}

## Radius diagnostics

- Pearson(radius, RMM amplitude) = **{r_p:.3f}** (p = {p_p:.2e})
- Radius across phases: ANOVA F = {f_ph:.2f}, p = {p_ph:.2e}
- Radius across ENSO: ANOVA F = {f_en:.2f}, p = {p_en:.2e}
- Mean radius: {radii_a.mean():.3f} ± {radii_a.std():.3f}
- Max norm during training: {max(history['max_norm']):.2f}

## Decision

{decision_text}

## Next

- `15_mjo_ssl_temporal_2d.ipynb` — SSL temporal encoder with 20–90 day bandpass
- `16_mjo_comparison.ipynb` — three-way comparison (RMM vs supervised vs SSL)
"""

with open(f'{RESULTS_DIR}/mjo_sup_summary.md', 'w') as f:
    f.write(summary)
print(f'\nSaved: {RESULTS_DIR}/mjo_sup_summary.md')

## Cell 13 — (Optional) Download Outputs

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(RESULTS_DIR)):
    files.download(f'{RESULTS_DIR}/{fname}')

---
## Done!

**Send back:**
1. `mjo_sup_summary.md` — auto-decision + comparison table
2. `embedding_2d_overview.png` — 4-panel scatter
3. `radius_diagnostics.png` — did the freed radius encode amplitude?
4. `training_curves.png` — sanity check on norms

Next: nb15 (SSL temporal 2D) — the central comparison notebook.

---
*DDCS Project | jh9141@nyu.edu*